In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F 
from pathlib import Path


In [4]:
spark = SparkSession.builder.master('local[*]').appName('step02').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/23 10:52:32 WARN Utils: Your hostname, DESKTOP-ED4F5AN, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/23 10:52:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/23 10:52:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.option('header',True).csv('/home/ilyes/spark-zoomcamp/fhvhv_tripdata_2021-01.csv.gz')

In [6]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [7]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [8]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(),True),
    types.StructField('dispatching_base_num',types.StringType(),True),
    types.StructField('pickup_datetime',types.TimestampType(),True),
    types.StructField('dropoff_datetime',types.TimestampType(),True),
    types.StructField('PULocationID',types.IntegerType(),True),
    types.StructField('DOLocationID',types.IntegerType(),True),
    types.StructField('SR_Flag',types.StringType(),True)
])

In [9]:
df.schema = schema

In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [12]:
df = spark.read.option('header',True).schema(schema).csv('/home/ilyes/spark-zoomcamp/fhvhv_tripdata_2021-01.csv.gz')

In [13]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [14]:
df = df.repartition(24)
df.write.parquet('/home/ilyes/spark-zoomcamp/notebook',mode='overwrite')

26/10/23 10:55:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/23 10:55:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/23 10:55:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [15]:
df = spark.read.parquet('/home/ilyes/spark-zoomcamp/data/fhvhv/2021/01')

In [16]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02617|2021-01-21 18:19:13|2021-01-21 19:14:27|         256|         265|   NULL|
|           HV0003|              B02871|2021-01-26 11:15:51|2021-01-26 11:23:14|         165|         165|   NULL|
|           HV0003|              B02764|2021-01-19 19:11:47|2021-01-19 19:29:45|          82|         137|   NULL|
|           HV0003|              B02872|2021-01-27 11:49:13|2021-01-27 11:53:15|          97|          97|   NULL|
|           HV0005|              B02510|2021-01-27 15:21:49|2021-01-27 15:29:59|         134|          95|   NULL|
|           HV0003|              B02765|2021-01-17 18:35:49|2021-01-17 18:55:43|

In [17]:
df_select = df.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID')

In [18]:
df_select.show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-21 18:19:13|2021-01-21 19:14:27|         256|         265|
|2021-01-26 11:15:51|2021-01-26 11:23:14|         165|         165|
|2021-01-19 19:11:47|2021-01-19 19:29:45|          82|         137|
|2021-01-27 11:49:13|2021-01-27 11:53:15|          97|          97|
|2021-01-27 15:21:49|2021-01-27 15:29:59|         134|          95|
|2021-01-17 18:35:49|2021-01-17 18:55:43|          74|         116|
|2021-01-28 05:46:58|2021-01-28 06:04:38|          78|          75|
|2021-01-04 11:08:55|2021-01-04 11:42:59|         236|         126|
|2021-01-21 09:38:00|2021-01-21 09:46:20|          79|         137|
|2021-01-29 08:33:14|2021-01-29 08:40:43|         198|         157|
|2021-01-24 18:44:55|2021-01-24 18:54:51|          37|          37|
|2021-01-05 18:13:18|2021-01-05 18:39:27|       

In [19]:
df.filter(df.hvfhs_license_num == 'HV0003').show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02617|2021-01-21 18:19:13|2021-01-21 19:14:27|         256|         265|   NULL|
|           HV0003|              B02871|2021-01-26 11:15:51|2021-01-26 11:23:14|         165|         165|   NULL|
|           HV0003|              B02764|2021-01-19 19:11:47|2021-01-19 19:29:45|          82|         137|   NULL|
|           HV0003|              B02872|2021-01-27 11:49:13|2021-01-27 11:53:15|          97|          97|   NULL|
|           HV0003|              B02765|2021-01-17 18:35:49|2021-01-17 18:55:43|          74|         116|   NULL|
|           HV0003|              B02872|2021-01-28 05:46:58|2021-01-28 06:04:38|

In [22]:
df.withColumn('pickup_date',F.to_date(df.pickup_datetime)).withColumn('dropoff_date',F.to_date(df.dropoff_datetime)).select('SR_Flag','DOLocationID','PULocationID','pickup_date','dropoff_date').show()

+-------+------------+------------+-----------+------------+
|SR_Flag|DOLocationID|PULocationID|pickup_date|dropoff_date|
+-------+------------+------------+-----------+------------+
|   NULL|         265|         256| 2021-01-21|  2021-01-21|
|   NULL|         165|         165| 2021-01-26|  2021-01-26|
|   NULL|         137|          82| 2021-01-19|  2021-01-19|
|   NULL|          97|          97| 2021-01-27|  2021-01-27|
|   NULL|          95|         134| 2021-01-27|  2021-01-27|
|   NULL|         116|          74| 2021-01-17|  2021-01-17|
|   NULL|          75|          78| 2021-01-28|  2021-01-28|
|   NULL|         126|         236| 2021-01-04|  2021-01-04|
|   NULL|         137|          79| 2021-01-21|  2021-01-21|
|   NULL|         157|         198| 2021-01-29|  2021-01-29|
|   NULL|          37|          37| 2021-01-24|  2021-01-24|
|   NULL|          72|         106| 2021-01-05|  2021-01-05|
|   NULL|         265|          35| 2021-01-20|  2021-01-20|
|   NULL|          39|  

In [23]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [24]:
t = crazy_stuff('B02884')
print(t)

s/b44


In [25]:
crazy_stuff_udf = F.udf(crazy_stuff,returnType=types.StringType())

/home/ilyes/spark-zoomcamp/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [32]:
df.withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)).select('base_id', 'pickup_datetime', 'dropoff_datetime','PULocationID', 'DOLocationID').show()

+-------+-------------------+-------------------+------------+------------+
|base_id|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------+-------------------+-------------------+------------+------------+
|  e/a39|2021-01-21 18:19:13|2021-01-21 19:14:27|         256|         265|
|  a/b37|2021-01-26 11:15:51|2021-01-26 11:23:14|         165|         165|
|  e/acc|2021-01-19 19:11:47|2021-01-19 19:29:45|          82|         137|
|  e/b38|2021-01-27 11:49:13|2021-01-27 11:53:15|          97|          97|
|  e/9ce|2021-01-27 15:21:49|2021-01-27 15:29:59|         134|          95|
|  s/acd|2021-01-17 18:35:49|2021-01-17 18:55:43|          74|         116|
|  e/b38|2021-01-28 05:46:58|2021-01-28 06:04:38|          78|          75|
|  a/a7a|2021-01-04 11:08:55|2021-01-04 11:42:59|         236|         126|
|  e/9ce|2021-01-21 09:38:00|2021-01-21 09:46:20|          79|         137|
|  e/b3c|2021-01-29 08:33:14|2021-01-29 08:40:43|         198|         157|
|  e/acc|202